In [2]:
import bz2
import os
from netCDF4 import Dataset

# Replace with your actual bz2 compressed NetCDF file path
bz2_file_path = "20170701.data.nc.bz2"
decompressed_nc_file_path = "sample.nc"  # Temporary decompressed file

# Step 1: Decompress and save as a NetCDF file
with bz2.BZ2File(bz2_file_path, 'rb') as bz2_file:
    with open(decompressed_nc_file_path, 'wb') as nc_file:
        nc_file.write(bz2_file.read())

# Step 2: Open the decompressed NetCDF file
with Dataset(decompressed_nc_file_path, mode='r') as nc_file:
    print("NetCDF File Metadata:")
    print(nc_file)

    print("\nDimensions:")
    for dim_name, dim in nc_file.dimensions.items():
        print(f"  {dim_name}: {len(dim)}")

    print("\nVariables:")
    for var_name, var in nc_file.variables.items():
        print(f"  {var_name}: {var.dtype}, Shape: {var.shape}")

    print("\nGlobal Attributes:")
    for attr_name in nc_file.ncattrs():
        print(f"  {attr_name}: {getattr(nc_file, attr_name)}")


NetCDF File Metadata:
<class 'netCDF4.Dataset'>
root group (NETCDF4 data model, file format HDF5):
    dimensions(sizes): 
    variables(dimensions): 
    groups: map, time_series

Dimensions:

Variables:

Global Attributes:


In [1]:
import bz2
import os
import tempfile
from netCDF4 import Dataset

# Replace with your actual .bz2 compressed NetCDF file path
bz2_file_path = "20170701.data.nc.bz2"  # Change this to your actual file path

# Check if the file exists
if not os.path.exists(bz2_file_path):
    raise FileNotFoundError(f"Error: File '{bz2_file_path}' not found.")

# Create a temporary file to store the decompressed NetCDF file
with tempfile.NamedTemporaryFile(suffix=".nc", delete=False) as temp_nc_file:
    decompressed_nc_file_path = temp_nc_file.name  # Store the path for later use

    # Decompress the .bz2 file and write it to the temporary NetCDF file
    try:
        with bz2.BZ2File(bz2_file_path, 'rb') as bz2_file:
            temp_nc_file.write(bz2_file.read())
    except Exception as e:
        raise RuntimeError(f"Error decompressing '{bz2_file_path}': {e}")

# Function to recursively explore NetCDF groups
def explore_group(nc_group, indent=0):
    """ Recursively print group structure, dimensions, variables, and attributes. """
    prefix = "  " * indent
    print(f"{prefix}📂 Group: {nc_group.path}")

    # Print dimensions
    if nc_group.dimensions:
        print(f"{prefix}  📏 Dimensions:")
        for dim_name, dim in nc_group.dimensions.items():
            print(f"{prefix}    - {dim_name}: {len(dim)}")

    # Print variables
    if nc_group.variables:
        print(f"{prefix}  📊 Variables:")
        for var_name, var in nc_group.variables.items():
            print(f"{prefix}    - {var_name}: {var.dtype}, Shape: {var.shape}")

    # Print attributes
    if nc_group.ncattrs():
        print(f"{prefix}  🏷️  Attributes:")
        for attr_name in nc_group.ncattrs():
            print(f"{prefix}    - {attr_name}: {getattr(nc_group, attr_name)}")

    # Recursively explore subgroups
    for grp_name, subgroup in nc_group.groups.items():
        explore_group(subgroup, indent + 1)

# Open and explore the NetCDF file
try:
    with Dataset(decompressed_nc_file_path, mode='r') as nc_file:
        print("\n🔍 NetCDF File Structure:\n")
        explore_group(nc_file)  # Start from the root group

except Exception as e:
    raise RuntimeError(f"Error reading NetCDF file '{decompressed_nc_file_path}': {e}")

finally:
    # Cleanup: Remove the temporary decompressed NetCDF file
    os.remove(decompressed_nc_file_path)
    print("\n✅ Cleanup: Temporary NetCDF file removed.")


🔍 NetCDF File Structure:

📂 Group: /
  📂 Group: /map
    📂 Group: /map/ut_hrs
      📏 Dimensions:
        - freq_MHz: 6
        - md_long: 361
        - md_lat: 181
        - ut_hrs: 144
      📊 Variables:
        - ut_sTime: int64, Shape: ()
        - freq_MHz: int64, Shape: (6,)
        - md_long: int64, Shape: (361,)
        - md_lat: int64, Shape: (181,)
        - ut_hrs: float64, Shape: (144,)
        - spot_density: float64, Shape: (6, 144, 361, 181)
    📂 Group: /map/slt_mid
      📏 Dimensions:
        - freq_MHz: 6
        - md_long: 361
        - md_lat: 181
        - slt_mid: 144
      📊 Variables:
        - ut_sTime: int64, Shape: ()
        - freq_MHz: int64, Shape: (6,)
        - md_long: int64, Shape: (361,)
        - md_lat: int64, Shape: (181,)
        - slt_mid: float64, Shape: (144,)
        - spot_density: float64, Shape: (6, 144, 361, 181)
  📂 Group: /time_series
    📂 Group: /time_series/ut_hrs
      📏 Dimensions:
        - freq_MHz: 6
        - ut_hrs: 145
      

In [3]:
import h5py

def load_hdf5(file_path):
    try:
        # Open the HDF5 file
        with h5py.File(file_path, 'r') as f:
            print(f"Contents of {file_path}:")
            
            # List all the keys (top-level groups/datasets)
            def print_attrs(name, obj):
                print(f"{name}: {obj}")
                
            # Iterate through all groups and datasets
            f.visititems(print_attrs)
            
            print("\nKeys in the file:")
            for key in f.keys():
                print(key)
            
            # Optionally, print detailed data for each dataset
            for key in f.keys():
                print(f"\nDataset '{key}':")
                print(f[key][:])  # Print the dataset's content (array)
    
    except Exception as e:
        print(f"Error reading the file: {e}")

# Example usage
file_path = 'data/spot_csvs/rsd2017-07-01.01.hdf5'  # Replace with your actual file path
load_hdf5(file_path)

Contents of data/spot_csvs/rsd2017-07-01.01.hdf5:
Data: <HDF5 group "/Data" (1 members)>
Data/Table Layout: <HDF5 dataset "Table Layout": shape (3239387,), type "|V167">
Metadata: <HDF5 group "/Metadata" (5 members)>
Metadata/Data Parameters: <HDF5 dataset "Data Parameters": shape (24,), type "|V110">
Metadata/Experiment Notes: <HDF5 dataset "Experiment Notes": shape (60,), type "|V80">
Metadata/Experiment Parameters: <HDF5 dataset "Experiment Parameters": shape (14,), type "|V51">
Metadata/Independent Spatial Parameters: <HDF5 dataset "Independent Spatial Parameters": shape (0,), type "|V58">
Metadata/_record_layout: <HDF5 dataset "_record_layout": shape (1,), type "|V24">

Keys in the file:
Data
Metadata

Dataset 'Data':
Error reading the file: Accessing a group is done with bytes or str, not <class 'slice'>


In [4]:
import pandas as pd

pd.read_hdf('data/spot_csvs/rsd2017-07-01.01.hdf5', key='Data/Table Layout')

: 

In [1]:
import dask.dataframe as dd
import pandas as pd

file_path = 'data/madrigal/rsd2017-07-01.01.hdf5'

# Read HDF5 file using Dask
df = dd.read_hdf(file_path, key='Data/Table Layout', chunksize=10000)  # Adjust chunksize as needed
df_sample = df.compute()
print(df_sample.head())

   year month day hour min sec  recno  kindat  kinst    ut1_unix  ...  \
0  2017    07  01   00  00  00      0   17578   8308  1498867200  ...   
1  2017    07  01   00  00  00      1   17578   8308  1498867200  ...   
2  2017    07  01   00  00  00      2   17578   8308  1498867200  ...   
3  2017    07  01   00  00  00      3   17578   8308  1498867200  ...   
4  2017    07  01   00  00  00      4   17578   8308  1498867200  ...   

   call_sign_rx      rxlat      rxlon       tfreq    sn   smode  ssrc  pthlen  \
0         KD6UY  31.104200 -97.958300  14076000.0   NaN  'JT65'   PSK  1832.5   
1        OH8GKP  64.812500  25.458333  10140191.0 -21.0    wspr   WSP  2109.0   
2        OE6RKE  46.812500  15.208333   7040120.0 -11.0    wspr   WSP  1264.0   
3        OE6PWD  47.062500  15.458333   7040107.0 -17.0    wspr   WSP  1267.0   
4         N4TVC  38.770833 -77.291667   7040111.0 -19.0    wspr   WSP  5889.0   

    latcen    loncen  
0  37.2199 -104.6411  
1  58.7724    9.8659  
2  49

In [2]:
df = df_sample
df['datetime'] = pd.to_datetime(df['year'] + '-' + df['month'] + '-' + df['day'] + ' ' + df['hour'] + ':' + df['min'] + ':' + df['sec'])
df.drop(['year', 'month', 'day', 'hour', 'min', 'sec'], axis=1, inplace=True)

In [3]:
print(df.head())

   year month day hour min sec  recno  kindat  kinst    ut1_unix  ...  \
0  2017    07  01   00  00  00      0   17578   8308  1498867200  ...   
1  2017    07  01   00  00  00      1   17578   8308  1498867200  ...   
2  2017    07  01   00  00  00      2   17578   8308  1498867200  ...   
3  2017    07  01   00  00  00      3   17578   8308  1498867200  ...   
4  2017    07  01   00  00  00      4   17578   8308  1498867200  ...   

       rxlat      rxlon       tfreq    sn   smode  ssrc  pthlen   latcen  \
0  31.104200 -97.958300  14076000.0   NaN  'JT65'   PSK  1832.5  37.2199   
1  64.812500  25.458333  10140191.0 -21.0    wspr   WSP  2109.0  58.7724   
2  46.812500  15.208333   7040120.0 -11.0    wspr   WSP  1264.0  49.3986   
3  47.062500  15.458333   7040107.0 -17.0    wspr   WSP  1267.0  49.5321   
4  38.770833 -77.291667   7040111.0 -19.0    wspr   WSP  5889.0  51.8668   

     loncen   datetime  
0 -104.6411 2017-07-01  
1    9.8659 2017-07-01  
2    7.6237 2017-07-01  
3   

In [1]:
import h5py
import dask.dataframe as dd
import dask.array as da
import numpy as np

STATIC_CHUNK_SIZE = 10_000

def load_hdf5_with_dask(file_path):
    
    dask_df = dd.read_hdf(file_path, '/Data/Table Layout', chunksize=STATIC_CHUNK_SIZE)
    
    for col in ['year', 'month', 'day', 'hour', 'min', 'sec']:
        dask_df[col] = dask_df[col].map(lambda x: x.decode('utf-8') if isinstance(x, bytes) else x, meta=(col, 'str'))

    dask_df['datetime'] = dd.to_datetime(dask_df['year'] + '-' + dask_df['month'] + '-' + dask_df['day'] + ' ' +
                                          dask_df['hour'] + ':' + dask_df['min'] + ':' + dask_df['sec'])

    dask_df = dask_df[['datetime', 'tfreq', 'pthlen', 'latcen', 'loncen']]
    dask_df = dask_df.rename(columns={'datetime': 'date', 'tfreq': 'freq', 'pthlen': 'dist_km', 
                                      'latcen': 'lat', 'loncen': 'long'})
    
    dask_df['freq'] = dask_df['freq'] * 1000

    return dask_df

file_path = "data/madrigal/rsd2017-07-01.01.hdf5"
dask_df = load_hdf5_with_dask(file_path)

df = dask_df.compute()
print(df.head())

KeyboardInterrupt: 